# 3 - A fast crash classifier (EfficientNet) to replace slow VLM scouting

The VLM scout needs about 4-9 s per frame. This notebook trains a small image classifier
(**EfficientNet-B0**, pretrained on ImageNet) that answers **crash / no crash in milliseconds**, even on CPU, so it
can look at *every* sampled frame - or run live.

* Trained on **all of dataset 1**; **dataset 2** (a different source) is cut in half: one half (`val`) picks two
  operating thresholds (best-F1, and a stricter one that keeps precision >= 0.90 if reachable), the other half
  (`test_EXTERNAL`) gives the final numbers. Near-copies of earlier data are removed from each evaluation set.
* With only one dataset available it falls back to a leaky split and says so - attach both.
* Saves PyTorch weights, a **TorchScript** file (no `timm` needed to run it) and a config with the thresholds.

**Time:** about 20-40 min on a T4. `QUICK = True` runs in ~3 min.

**Honest limits.** A frame classifier sees one picture, not motion: it recognises *crash scenes* (wreckage,
contact, smoke), not the moment of impact. Expect it to be weaker on views unlike its training data.

### How to run on Kaggle
1. **Settings (right panel):** Accelerator = **GPU T4 x2** (or P100), **Internet = On**.
2. **Run All** to test, or **Save Version -> Save & Run All (Commit)** so the outputs survive after the session.
3. Everything you need to keep is written to `/kaggle/working/outputs/` (also zipped at the end). Download it from the notebook's **Output** tab.


In [ ]:
!pip install -q -U kagglehub timm pillow scikit-learn pandas
import os, sys, json, time, random, re, glob, math, shutil
from pathlib import Path

ROOT = Path(os.environ.get("WORK", "/kaggle/working"))   # Kaggle's writable folder
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)

# Kaggle stops a session after 12 h. T0 is the clock; every long step checks hours_left().
T0 = time.time()
HARD_LIMIT_H = 11.0          # leave a 1 h margin below Kaggle's 12 h


def hours_left(reserve_h=0.0):
    """Hours still available before the HARD_LIMIT_H deadline, minus a reserve for later steps."""
    return HARD_LIMIT_H - (time.time() - T0) / 3600 - reserve_h

import torch, timm
print("torch", torch.__version__, "timm", timm.__version__, "| cuda:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# ---------------- settings ----------------
QUICK = False
ARCH = "efficientnet_b0"
IMG = 256
EPOCHS = 2 if QUICK else 8
BATCH = 32 if QUICK else 64
LR = 3e-4
N_TRAIN_PER_CLASS = 100 if QUICK else 4000
N_EVAL_PER_CLASS = 60 if QUICK else 1500
WORKERS = 2


In [ ]:
# ---------------- data: download, label from folder names, split ----------------
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

np.random.seed(SEED)

# Datasets to try, in priority order. The FIRST one that downloads is used for train/val;
# the next one(s) become an EXTERNAL test set. A different source is the honest test:
# frames from the same video never end up on both sides of it.
CANDIDATES = [
    "suryaprabhakaran2005/road-accidents-from-cctv-footages-dataset",
    "ckay16/accident-detection-from-cctv-footage",
    "umitka/real-world-vehicle-crash-dataset-balanced",
]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NEG = re.compile(r"(non[\s_-]*accident|no[\s_-]*accident|not[\s_-]*accident|non[\s_-]*crash|"
                 r"no[\s_-]*crash|normal|safe|negative)", re.I)
POS = re.compile(r"(accident|crash|collision|positive)", re.I)


def label_of(rel_parts):
    """1 = crash, 0 = normal, None = cannot tell. Looks at folder names, deepest first."""
    for part in reversed(rel_parts[:-1]):
        if NEG.search(part):      # checked first: "Non Accident" also contains "accident"
            return 0
        if POS.search(part):
            return 1
    return None


def scan(root, name):
    rows, skipped = [], []
    root = Path(root)
    for p in root.rglob("*"):
        if p.suffix.lower() not in IMG_EXT:
            continue
        rel = p.relative_to(root).parts
        y = label_of(rel)
        if y is None:
            skipped.append(str(p))
            continue
        video = re.sub(r"[_\-\s]*\d+$", "", p.stem)       # drop only the LAST number (the frame no.) -> video id
        rows.append(dict(path=str(p), label=y, source=name,
                         group=f"{name}|{p.parent.relative_to(root)}|{video}"))
    return pd.DataFrame(rows), skipped


def fetch(slug):
    import kagglehub
    return kagglehub.dataset_download(slug)


sources = []                                   # (name, root) pairs
for slug in CANDIDATES:
    try:
        sources.append((slug, fetch(slug)))
    except Exception as e:                    # not found / no internet / needs login
        print(f"skip {slug}: {type(e).__name__}: {str(e)[:90]}")
for d in sorted(glob.glob("/kaggle/input/*")):  # datasets attached through "Add data"
    if re.search("accident|crash", d, re.I) and d not in [r for _, r in sources]:
        sources.append((Path(d).name, d))

frames = []
for name, root in sources:
    df, skipped = scan(root, name)
    npos = int(df.label.sum()) if len(df) else 0
    print(f"{name}: {len(df)} labelled images ({npos} crash / {len(df) - npos} normal), "
          f"{len(skipped)} skipped (label unclear)")
    if skipped:
        print("   e.g. skipped:", skipped[:2])
    if len(df) and df.label.nunique() == 2:
        frames.append(df)
assert frames, ("No usable dataset found. Turn Internet ON in the notebook settings, or use "
                "'Add data' to attach one of CANDIDATES, or edit CANDIDATES.")


def split(df, test_size):
    if df.group.nunique() >= 20:                         # video ids recovered: split by video
        a, b = next(GroupShuffleSplit(1, test_size=test_size, random_state=SEED)
                    .split(df, groups=df.group))
        return df.iloc[a], df.iloc[b], True
    print("WARNING: could not recover video ids from the file names -> random frame split. "
          "Frames of one video can land on both sides, so val numbers are OPTIMISTIC; "
          "trust the EXTERNAL test instead.")
    a, b = train_test_split(df.index, test_size=test_size, stratify=df.label, random_state=SEED)
    return df.loc[a], df.loc[b], False


main = frames[0]
if len(frames) > 1:
    # PROTOCOL: train on ALL of dataset 1. Dataset 2 (a different source) is cut in half:
    #   val  -> tune the decision thresholds,   test_ext -> the final numbers, never used for tuning.
    # (Dataset 1 is made of near-identical video frames, so a split of it cannot be a fair test.)
    train, test_in = main, None
    val, test_ext, grouped = split(pd.concat(frames[1:]).reset_index(drop=True), 0.5)
    print("PROTOCOL: train = dataset 1 | val (thresholds) = half of dataset 2 | test = other half of dataset 2")
else:
    print("WARNING: only ONE dataset is available. Falling back to splitting it; its frames come from the same\n"
          "videos, so val/test numbers will be OPTIMISTIC. Attach a second crash dataset for an honest test.")
    trainval, test_in, grouped = split(main, 0.15)
    train, val, _ = split(trainval, 0.15)
    test_ext = None


def cap(df, n_per_class, seed=SEED):
    """Balanced subsample: at most n_per_class of each label."""
    parts = [g.sample(min(len(g), n_per_class), random_state=seed) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


def summary(name, df):
    if df is None:
        print(f"{name:14s} none (only one dataset found - there is NO external test)")
    else:
        print(f"{name:14s} {len(df):6d} images  crash={int(df.label.sum()):6d}  "
              f"normal={int((1 - df.label).sum()):6d}  sources={sorted(df.source.unique())}")


for n_, d_ in (("train", train), ("val", val), ("test (same src)", test_in), ("test EXTERNAL", test_ext)):
    if d_ is not None or n_ == "test EXTERNAL":
        summary(n_, d_)


# ---- near-duplicate guard: no evaluation image may be a copy of a training image ----------
# Two public crash datasets are often cut from the same source videos, which would make an
# "external" test quietly optimistic. dHash = 64-bit fingerprint, robust to resizing / JPEG.
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def dhash(path):
    try:
        im = Image.open(path).convert("L").resize((9, 8), Image.BILINEAR)
    except Exception:
        return None
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int.from_bytes(np.packbits(bits).tobytes(), "big")


def hashes_of(df):
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(dhash, df.path))


def drop_near_duplicates(eval_df, ref_hashes, name, max_dist=6):
    """Remove eval images within `max_dist` bits of any training image (and unreadable files)."""
    ref = np.array([h for h in ref_hashes if h is not None], dtype=np.uint64)
    keep = []
    for h in hashes_of(eval_df):
        if h is None:
            keep.append(False)
            continue
        d = POP[np.bitwise_xor(ref, np.uint64(h)).view(np.uint8)].reshape(-1, 8).sum(axis=1).min()
        keep.append(bool(d > max_dist))
    out = eval_df[np.array(keep)].reset_index(drop=True)
    n_rm = len(eval_df) - len(out)
    print(f"{name}: removed {n_rm} of {len(eval_df)} images that are near-copies of training images")
    if len(eval_df) and n_rm / len(eval_df) > 0.25:
        print(f"   WARNING: {name} overlaps heavily with the training data - the datasets share source "
              f"videos. Prefer a different external dataset; treat this test set's numbers with care.")
    return out


def make_eval_sets(train_hashes, n_per_class):
    """Evaluation sets with every near-copy of earlier data removed, in order: train -> val -> test."""
    sets = {"val": drop_near_duplicates(cap(val, n_per_class), train_hashes, "val")}
    ref = list(train_hashes) + hashes_of(sets["val"])          # the test set must not repeat val either
    if test_ext is not None:
        sets["test_EXTERNAL"] = drop_near_duplicates(cap(test_ext, n_per_class), ref, "test_EXTERNAL")
    if test_in is not None:
        sets["test_same_source"] = drop_near_duplicates(cap(test_in, n_per_class), ref, "test_same_source")
    for k, v in sets.items():
        if len(v) < 40:
            print(f"WARNING: '{k}' has only {len(v)} images after cleaning - its numbers are not reliable.")
    return sets


In [ ]:
# ---------------- datasets and model ----------------
from PIL import Image
import torchvision.transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_train = T.Compose([T.RandomResizedCrop(IMG, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_eval = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])


class Frames(torch.utils.data.Dataset):
    def __init__(self, df, tf):
        self.paths, self.labels, self.tf = list(df.path), list(df.label), tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG, IMG))                 # unreadable file: neutral grey-ish image
        return self.tf(img), torch.tensor(float(self.labels[i]))


def loader(df, tf, shuffle):
    return torch.utils.data.DataLoader(Frames(df, tf), batch_size=BATCH, shuffle=shuffle,
                                       num_workers=WORKERS, pin_memory=(DEVICE == "cuda"))


train_df = cap(train, N_TRAIN_PER_CLASS)
print("fingerprinting the training images for the duplicate check ...")
train_hashes = hashes_of(train_df)
eval_sets = make_eval_sets(train_hashes, N_EVAL_PER_CLASS)

model = timm.create_model(ARCH, pretrained=True, num_classes=1).to(DEVICE)
print("parameters: %.1f M" % (sum(p.numel() for p in model.parameters()) / 1e6))


In [ ]:
# ---------------- evaluation helper (same code as in the other notebooks) ----------------
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix


def metrics_at(y, score, thr):
    pred = (np.asarray(score) >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(precision=round(float(p), 3), recall=round(float(r), 3), f1=round(float(f), 3),
                tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn), thr=float(thr))


def pick_thresholds(y, score):
    """Two operating points chosen on VALIDATION data only:
       best-F1, and the lowest threshold that keeps precision >= 0.90 (fewer false alarms)."""
    y = np.asarray(y); score = np.asarray(score)
    grid = np.unique(np.quantile(score, np.linspace(0.01, 0.99, 197)))
    best_f1 = max(grid, key=lambda t: metrics_at(y, score, t)["f1"])
    ok = [t for t in grid if metrics_at(y, score, t)["precision"] >= 0.90]
    if not ok:
        print("NOTE: precision 0.90 is not reachable on the validation data - 'precision90' falls back to "
              "the best-F1 threshold, so it is NOT a high-precision operating point here.")
    return dict(best_f1=float(best_f1), precision90=float(min(ok)) if ok else float(best_f1))


def report(name, y, score, thr):
    out = {"n": int(len(y))}
    try:
        out["auc"] = round(float(roc_auc_score(y, score)), 3)
    except ValueError:
        out["auc"] = None
    for k, t in thr.items():
        out[k] = metrics_at(y, score, t)
    print(name, json.dumps(out))
    return out


In [ ]:
# ---------------- train ----------------
dl = loader(train_df, tf_train, True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.15)
scaler = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == "cuda"))
lossf = torch.nn.BCEWithLogitsLoss()


@torch.no_grad()
def scores_of(df):
    model.eval(); out = []
    for x, _ in loader(df, tf_eval, False):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=(DEVICE == "cuda")):
            out.append(model(x.to(DEVICE)).squeeze(1).float().cpu())
    return torch.cat(out).numpy()


history, t0, bad = [], time.time(), 0
for ep in range(EPOCHS):
    if ep > 0 and hours_left(0.4) < 0:                          # Kaggle's 12 h limit is near
        print("[time guard] out of time - keeping the best epoch so far")
        break
    model.train(); losses = []
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=(DEVICE == "cuda")):
            loss = lossf(model(x).squeeze(1), y)
        if not torch.isfinite(loss):                            # skip a bad batch; stop if it persists
            bad += 1
            print(f"non-finite loss ({bad} so far) - batch skipped")
            if bad >= 20:
                raise RuntimeError("Training diverged (20 non-finite losses): lower LR and run again.")
            continue
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        losses.append(loss.item())
    s = scores_of(eval_sets["val"]); y = eval_sets["val"].label.values
    auc = roc_auc_score(y, s)
    history.append(dict(epoch=ep + 1, train_loss=float(np.mean(losses)), val_auc=float(auc)))
    print(f"epoch {ep + 1}/{EPOCHS}  train loss {np.mean(losses):.4f}  val AUC {auc:.3f}  "
          f"({(time.time() - t0) / 60:.1f} min)")
    if auc >= max(h["val_auc"] for h in history):
        torch.save(model.state_dict(), OUT / "crash_classifier_best.pt")      # keep the best epoch
model.load_state_dict(torch.load(OUT / "crash_classifier_best.pt", map_location=DEVICE))


In [ ]:
# ---------------- thresholds from VAL only; report all sets ----------------
all_scores = {n: scores_of(d) for n, d in eval_sets.items()}
thr = pick_thresholds(eval_sets["val"].label.values, all_scores["val"])
print("thresholds (logit scale, chosen on validation):", thr)
results = {n: report(n, d.label.values, all_scores[n], thr) for n, d in eval_sets.items()}


In [ ]:
# ---------------- speed ----------------
model.eval()
x = torch.randn(1, 3, IMG, IMG).to(DEVICE)
with torch.no_grad():
    for _ in range(5): model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(50): model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
ms_gpu = (time.time() - t) / 50 * 1000
cpu_model = timm.create_model(ARCH, pretrained=False, num_classes=1)
cpu_model.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()}); cpu_model.eval()
xc = torch.randn(1, 3, IMG, IMG)
with torch.no_grad():
    cpu_model(xc); t = time.time()
    for _ in range(20): cpu_model(xc)
ms_cpu = (time.time() - t) / 20 * 1000
print(f"per-image latency: {ms_gpu:.1f} ms on {DEVICE}, {ms_cpu:.0f} ms on CPU")


In [ ]:
# ---------------- save everything needed to run it elsewhere ----------------
scripted = torch.jit.trace(cpu_model, xc)
scripted.save(str(OUT / "crash_classifier_ts.pt"))
cfg = dict(arch=ARCH, img_size=IMG, mean=MEAN, std=STD, output="single logit; crash if logit >= threshold",
           thresholds=thr, default_threshold=thr["precision90"], datasets=[n for n, _ in sources],
           grouped_split=bool(grouped), latency_ms=dict(gpu=round(ms_gpu, 1), cpu=round(ms_cpu)),
           history=history, results=results,
           note="Judge it on test_EXTERNAL (a different dataset), not on val.")
(OUT / "crash_classifier_config.json").write_text(json.dumps(cfg, indent=2))
(OUT / "HOW_TO_USE_CLASSIFIER.txt").write_text("""import torch, json, cv2, numpy as np
cfg = json.load(open("crash_classifier_config.json"))
net = torch.jit.load("crash_classifier_ts.pt").eval()          # no timm needed
def crash_score(bgr):                                          # bgr: OpenCV frame
    x = cv2.cvtColor(cv2.resize(bgr, (cfg["img_size"],) * 2), cv2.COLOR_BGR2RGB) / 255.0
    x = (x - np.array(cfg["mean"])) / np.array(cfg["std"])
    x = torch.tensor(x.transpose(2, 0, 1)[None], dtype=torch.float32)
    with torch.no_grad():
        return net(x).item()                                   # logit; crash if >= cfg["default_threshold"]
""")
print(json.dumps({k: cfg[k] for k in ("thresholds", "latency_ms")}, indent=1))
shutil.make_archive(str(ROOT / "classifier_outputs"), "zip", OUT)
print("Download:", ROOT / "classifier_outputs.zip", "->", sorted(p.name for p in OUT.iterdir()))
